# Categories of AI apps

Run `4-classify-job-postings.ipynb` first so that `jobs/2-classified-jobs.jsonl` exists.

This notebook reads the LLM-filtered AI engineer job postings and uses `gpt-5.4-mini` to summarize the common AI application use cases these companies are building.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

## Load the Classified Jobs

This cell reads the JSONL file created by `4-classify-job-postings.ipynb`.

If the file is missing, run the classification notebook first.

In [ ]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

if not classified_jobs_path.exists():
    raise FileNotFoundError(
        f"Classified jobs JSONL file not found: {classified_jobs_path.resolve()}. Run 4-classify-job-postings.ipynb first."
    )

if classified_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The classified jobs JSONL file is empty. Run 4-classify-job-postings.ipynb first."
    )

jobs_for_summary = pd.read_json(classified_jobs_path, lines=True)

print(f"Loaded {len(jobs_for_summary)} job postings")

## Keep Only AI Engineering Roles

The classified jobs file contains all postings, including the ones the model rejected.

This cell keeps only the rows where `is_ai_engineering_role` is `True`.

In [ ]:
# Keep only the jobs the LLM classified as true AI engineering roles
jobs_for_summary = jobs_for_summary[jobs_for_summary["is_ai_engineering_role"]]

print(f"Loaded {len(jobs_for_summary)} classified AI engineering job postings")

## Combine All Descriptions

We merge all job descriptions into one large string so we can send them to the model in a single request.

Each description is separated by `---` so the model can tell where one posting ends and the next begins.

In [ ]:
descriptions = jobs_for_summary["description"].astype(str)

job_descriptions = "\n\n---\n\n".join(description for description in descriptions)

print(f"Summarizing categories across {len(descriptions)} job postings")

## Prepare the Prompt

We combine the job descriptions into one prompt and ask the model to identify the top five categories of AI applications those companies are building.

In [ ]:
prompt = f"""
Read the job descriptions below and identify the Top 5 categories of AI applications that companies are building.

Return markdown in exactly this format:
### Top 5 Categories of AI Applications
1. **[Category name]** — one sentence describing the category, followed by a concrete real-world example from the job descriptions (e.g. "Notion uses this to let teams search and automate work across their documents")
2. ...
(continue through 5)

Rank from most to least commonly seen across all job descriptions.

Requirements:
- Each category must include at least one specific, real product example drawn directly from the descriptions
- Use plain, concrete language — avoid enterprise jargon
- Focus on what the AI actually does for the user, not the underlying technology
- Do not mention hiring or recruiting details

Job descriptions:
{job_descriptions}
"""

## Summarize the Categories

Now we send the combined prompt to the model and display the response.

The model reads all job descriptions at once and returns the top five categories of AI applications those companies are building.

In [ ]:
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4-mini",
    input=prompt,
)

display(Markdown(response.output_text))